# Getting Started with Business Ontology

This notebook is Part 2 of 2 for the [Getting Started with Business Ontology](https://www.snowflake.com/en/developers/guides/getting-started-with-business-ontology/) guide. It resumes after the indexing wait at the end of Part 1 and follows Steps 8 to 10 of the guide:

- confirms that Business Ontology search can find your new terms
- creates 7 agents, including `ENRICHED_AGENT` (semantic views only) and `BON_AGENT` (the same views plus ontology search)
- asks both agents the same questions, to show the difference the ontology makes

Run the cells in order, top to bottom.

## Resume the Session

This is a new session, so this cell sets the database, schema and warehouse from Part 1 as the current context.

In [ ]:
%%sql -r context
USE DATABASE BUSINESS_ONTOLOGY;
USE SCHEMA SAP;
USE WAREHOUSE BUSINESS_ONTOLOGY_WH;

## 8. Confirm Search Indexing

Before an agent can use the ontology, the terms have to appear in search. Search is checked from a terminal with the Cortex Code CLI rather than from a notebook cell:

```bash
cortex search object "supplier count" --types=business-ontology
cortex search object "tier a suppliers" --types=business-ontology
```

**Check:** results show `Procurement.Supplier Count` and `Procurement.Tier A Supplier`, with their kind and definition. If they don't, check that the terms are approved, that your role can read the domain, and that enough time has passed. Wait until both searches pass before you ask the agents anything in Step 10.

## 9. Build the Agent

Once search can find your terms, you create the agents that put the ontology to work. This cell creates 7 agents with the same model, budget and core instructions. Agents with Business Ontology (BON) search turned on have `BON` in their name. The A/B comparison uses two of them: `ENRICHED_AGENT`, which queries the three enriched semantic views, and `BON_AGENT`, which adds a `business_ontology` search tool and instructions to search it before querying data. The other five are used only by the full benchmark.

In [ ]:
%%sql -r agents
-- =====================================================================
-- Business Ontology demo, 09: Agent matrix
-- Every agent: same model (auto), same budget (150 s / 40k tokens), same
-- response text, same domain tool descriptions. The ONLY difference between
-- an arm and its "+ BON" twin is the business_ontology tool and its 2-step instruction.
--
--   Arm A  BARE_AGENT        3 BARE SVs
--   Arm B  ENRICHED_AGENT    3 ENRICHED SVs
--   Arm C  BON_AGENT         3 ENRICHED SVs + Business Ontology      (BON uplift = C - B)
--   Consistency pairs (do two teams get the same answer?)
--          FINANCE_ENRICHED_AGENT / FINANCE_BON_AGENT   Finance + Procurement ENRICHED SVs
--          SALES_ENRICHED_AGENT   / SALES_BON_AGENT     Sales + Procurement ENRICHED SVs
--   Finance sees active_supplier_count (14); Sales only sees vendor_record_count (15).
--   Without BON they disagree; with BON both should answer 14.
--
-- After creating, DESCRIBE AGENT and confirm agent_spec contains
--   "EnableSnowscopeBusinessOntologySearch":true, "type":"snowscope_search", "corpus":"businessOntology"
-- (CREATE AGENT does not validate these values.)
-- BON search index refresh can take up to 6 hours after ontology edits or grants.
-- =====================================================================

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.BARE_AGENT
  COMMENT = 'benchmark arm A: 3 BARE domain semantic views, no Business Ontology'
  PROFILE = '{"display_name": "SAP Assistant (bare SVs)"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
  orchestration: |
    Data lives in three domain tools. Each tool queries one domain only and
    cannot join to another domain, so for questions that span domains, query
    each relevant tool and combine the results yourself.
    - query_procurement: vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials
    - query_finance: accounts payable postings, COGS, cost allocations by cost object
    - query_sales: sales orders, pricing conditions, discounts, customers
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_finance
      description: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_sales
      description: Sales domain data - sales orders, pricing conditions, discounts, customers.
tool_resources:
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_BARE_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_finance:
    semantic_view: BUSINESS_ONTOLOGY.SAP.FINANCE_BARE_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_sales:
    semantic_view: BUSINESS_ONTOLOGY.SAP.SALES_BARE_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.ENRICHED_AGENT
  COMMENT = 'benchmark arm B: 3 ENRICHED domain semantic views, no Business Ontology'
  PROFILE = '{"display_name": "SAP Assistant (enriched SVs)"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
  orchestration: |
    Data lives in three domain tools. Each tool queries one domain only and
    cannot join to another domain, so for questions that span domains, query
    each relevant tool and combine the results yourself.
    - query_procurement: vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials
    - query_finance: accounts payable postings, COGS, cost allocations by cost object
    - query_sales: sales orders, pricing conditions, discounts, customers
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_finance
      description: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_sales
      description: Sales domain data - sales orders, pricing conditions, discounts, customers.
tool_resources:
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_finance:
    semantic_view: BUSINESS_ONTOLOGY.SAP.FINANCE_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_sales:
    semantic_view: BUSINESS_ONTOLOGY.SAP.SALES_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.BON_AGENT
  COMMENT = 'benchmark arm C: 3 ENRICHED domain semantic views + native Business Ontology search'
  PROFILE = '{"display_name": "SAP Assistant (enriched SVs + Business Ontology)"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
experimental:
  EnableSnowscopeBusinessOntologySearch: true
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
    When you used a governed Business Ontology definition, name the term and
    the semantic view metric that implements it.
  orchestration: |
    Data lives in three domain tools. Each tool queries one domain only and
    cannot join to another domain, so for questions that span domains, query
    each relevant tool and combine the results yourself.
    - query_procurement: vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials
    - query_finance: accounts payable postings, COGS, cost allocations by cost object
    - query_sales: sales orders, pricing conditions, discounts, customers

    Before querying data, call business_ontology to look up the governed
    definition of each business term in the question. Prefer terms from the
    Procurement, Finance & Controlling, and Sales & Distribution domains. When
    the ontology names a canonical metric and the semantic view metric that
    implements it, use that metric rather than a similarly named variant, and
    follow any policy that applies to it. If the question only asks what
    something means, answer from the ontology without querying data.
tools:
  - tool_spec:
      type: snowscope_search
      name: business_ontology
      description: Governed business definitions - canonical metrics and their implementing semantic view metrics, scoped variants, business policies, code meanings, and how concepts relate across Procurement, Finance and Sales.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_finance
      description: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_sales
      description: Sales domain data - sales orders, pricing conditions, discounts, customers.
tool_resources:
  business_ontology:
    corpus: businessOntology
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_finance:
    semantic_view: BUSINESS_ONTOLOGY.SAP.FINANCE_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_sales:
    semantic_view: BUSINESS_ONTOLOGY.SAP.SALES_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

-- ------------------------------------------------ Consistency pairs
CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.FINANCE_ENRICHED_AGENT
  COMMENT = 'consistency pair: Finance team agent (Finance + Procurement enriched SVs), no BON'
  PROFILE = '{"display_name": "Finance Assistant"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
  orchestration: |
    You support the Finance team. Data lives in the domain tools below. Each
    tool queries one domain only and cannot join to another domain, so for
    questions that span domains, query each relevant tool and combine the
    results yourself.
    - query_finance: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
    - query_procurement: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_finance
      description: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tool_resources:
  query_finance:
    semantic_view: BUSINESS_ONTOLOGY.SAP.FINANCE_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.FINANCE_BON_AGENT
  COMMENT = 'consistency pair: Finance team agent (Finance + Procurement enriched SVs) + BON'
  PROFILE = '{"display_name": "Finance Assistant + BON"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
experimental:
  EnableSnowscopeBusinessOntologySearch: true
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
    When you used a governed Business Ontology definition, name the term and
    the semantic view metric that implements it.
  orchestration: |
    You support the Finance team. Data lives in the domain tools below. Each
    tool queries one domain only and cannot join to another domain, so for
    questions that span domains, query each relevant tool and combine the
    results yourself.
    - query_finance: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
    - query_procurement: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
    Before querying data, call business_ontology to look up the governed
    definition of each business term in the question. Prefer terms from the
    Procurement, Finance & Controlling, and Sales & Distribution domains. When
    the ontology names a canonical metric and the semantic view metric that
    implements it, use that metric rather than a similarly named variant, and
    follow any policy that applies to it. If the ontology names a semantic view
    metric this agent does not have, compute the same definition with the
    tools available. If the question only asks what something means, answer
    from the ontology without querying data.
tools:
  - tool_spec:
      type: snowscope_search
      name: business_ontology
      description: Governed business definitions - canonical metrics and their implementing semantic view metrics, scoped variants, business policies, code meanings, and how concepts relate across Procurement, Finance and Sales.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_finance
      description: Finance domain data - accounts payable postings, cost of goods sold, cost allocations by cost object.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tool_resources:
  business_ontology:
    corpus: businessOntology
  query_finance:
    semantic_view: BUSINESS_ONTOLOGY.SAP.FINANCE_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.SALES_ENRICHED_AGENT
  COMMENT = 'consistency pair: Sales team agent (Sales + Procurement enriched SVs), no BON'
  PROFILE = '{"display_name": "Sales Assistant"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
  orchestration: |
    You support the Sales team. Data lives in the domain tools below. Each
    tool queries one domain only and cannot join to another domain, so for
    questions that span domains, query each relevant tool and combine the
    results yourself.
    - query_sales: Sales domain data - sales orders, pricing conditions, discounts, customers.
    - query_procurement: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_sales
      description: Sales domain data - sales orders, pricing conditions, discounts, customers.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tool_resources:
  query_sales:
    semantic_view: BUSINESS_ONTOLOGY.SAP.SALES_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

CREATE OR REPLACE AGENT BUSINESS_ONTOLOGY.SAP.SALES_BON_AGENT
  COMMENT = 'consistency pair: Sales team agent (Sales + Procurement enriched SVs) + BON'
  PROFILE = '{"display_name": "Sales Assistant + BON"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
experimental:
  EnableSnowscopeBusinessOntologySearch: true
orchestration:
  budget:
    seconds: 150
    tokens: 40000
instructions:
  response: |
    You are an enterprise analytics assistant for SAP supply chain data.
    Answer precisely. State the final answer as a clear number or list first,
    then briefly explain how it was computed and which data was used.
    When you used a governed Business Ontology definition, name the term and
    the semantic view metric that implements it.
  orchestration: |
    You support the Sales team. Data lives in the domain tools below. Each
    tool queries one domain only and cannot join to another domain, so for
    questions that span domains, query each relevant tool and combine the
    results yourself.
    - query_sales: Sales domain data - sales orders, pricing conditions, discounts, customers.
    - query_procurement: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
    Before querying data, call business_ontology to look up the governed
    definition of each business term in the question. Prefer terms from the
    Procurement, Finance & Controlling, and Sales & Distribution domains. When
    the ontology names a canonical metric and the semantic view metric that
    implements it, use that metric rather than a similarly named variant, and
    follow any policy that applies to it. If the ontology names a semantic view
    metric this agent does not have, compute the same definition with the
    tools available. If the question only asks what something means, answer
    from the ontology without querying data.
tools:
  - tool_spec:
      type: snowscope_search
      name: business_ontology
      description: Governed business definitions - canonical metrics and their implementing semantic view metrics, scoped variants, business policies, code meanings, and how concepts relate across Procurement, Finance and Sales.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_sales
      description: Sales domain data - sales orders, pricing conditions, discounts, customers.
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: query_procurement
      description: Procurement domain data - vendors, materials, purchase orders, shipments, carriers, inspections, bill of materials.
tool_resources:
  business_ontology:
    corpus: businessOntology
  query_sales:
    semantic_view: BUSINESS_ONTOLOGY.SAP.SALES_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
  query_procurement:
    semantic_view: BUSINESS_ONTOLOGY.SAP.PROCUREMENT_ENRICHED_SV
    execution_environment: {type: warehouse, warehouse: BUSINESS_ONTOLOGY_WH, query_timeout: 120}
$$;

**Check:** `CREATE AGENT` doesn't validate the ontology settings, and a typo means the agent silently never searches the ontology. This cell reads the spec back from `BON_AGENT`. Expected: the `agent_spec` column contains `EnableSnowscopeBusinessOntologySearch` set to `true`, a tool of type `snowscope_search`, and `"corpus":"businessOntology"`.

In [ ]:
%%sql -r check_agent
DESCRIBE AGENT BUSINESS_ONTOLOGY.SAP.BON_AGENT;

## 10. Run A/B Tests

Everything is built, so you can now ask two otherwise identical agents the same questions. Each question has an answer the ontology makes official and a look-alike an agent without it may give:

| Question | Correct answer | Defensible-but-wrong answer |
|---|---|---|
| How many suppliers do we have? | 14 (active suppliers) | 15 (all vendor records, including probationary) |
| What is our revenue? | 711,915 (net of K007 discounts) | 746,000 (gross bookings) |

Start with `ENRICHED_AGENT`, which has no ontology. This cell asks it the supplier question. The agent call sits inside a short scripting block because `DATA_AGENT_RUN` needs its arguments as constants; the block then keeps only the text of the reply.

In [ ]:
%%sql -r enriched_suppliers
EXECUTE IMMEDIATE $$
DECLARE r VARCHAR;
BEGIN
  r := (SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('BUSINESS_ONTOLOGY.SAP.ENRICHED_AGENT', '{"messages": [{"role": "user", "content": [{"type": "text", "text": "How many suppliers do we have?"}]}]}'));
  RETURN (SELECT LISTAGG(c.value:text::STRING, '\n') FROM TABLE(FLATTEN(PARSE_JSON(:r):content)) c WHERE c.value:type = 'text');
END;
$$;

This cell asks `ENRICHED_AGENT` the revenue question.

In [ ]:
%%sql -r enriched_revenue
EXECUTE IMMEDIATE $$
DECLARE r VARCHAR;
BEGIN
  r := (SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('BUSINESS_ONTOLOGY.SAP.ENRICHED_AGENT', '{"messages": [{"role": "user", "content": [{"type": "text", "text": "What is our revenue?"}]}]}'));
  RETURN (SELECT LISTAGG(c.value:text::STRING, '\n') FROM TABLE(FLATTEN(PARSE_JSON(:r):content)) c WHERE c.value:type = 'text');
END;
$$;

Now ask `BON_AGENT` the same two questions. This cell asks the supplier question.

In [ ]:
%%sql -r bon_suppliers
EXECUTE IMMEDIATE $$
DECLARE r VARCHAR;
BEGIN
  r := (SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('BUSINESS_ONTOLOGY.SAP.BON_AGENT', '{"messages": [{"role": "user", "content": [{"type": "text", "text": "How many suppliers do we have?"}]}]}'));
  RETURN (SELECT LISTAGG(c.value:text::STRING, '\n') FROM TABLE(FLATTEN(PARSE_JSON(:r):content)) c WHERE c.value:type = 'text');
END;
$$;

This cell asks `BON_AGENT` the revenue question.

**Check:** `BON_AGENT` gives the correct answers and names the governed term and metric it used, for example `Procurement.Supplier Count` implemented by `ACTIVE_SUPPLIER_COUNT`. `ENRICHED_AGENT` may give the right number on some runs, but it can also pick the look-alike or hedge, and it has no governed term to cite. Agents vary from run to run, so rerun each cell two or three times. To measure the difference across all 20 questions, run `benchmark/run_benchmark.py` from the companion repository, as described in the guide.

In [ ]:
%%sql -r bon_revenue
EXECUTE IMMEDIATE $$
DECLARE r VARCHAR;
BEGIN
  r := (SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('BUSINESS_ONTOLOGY.SAP.BON_AGENT', '{"messages": [{"role": "user", "content": [{"type": "text", "text": "What is our revenue?"}]}]}'));
  RETURN (SELECT LISTAGG(c.value:text::STRING, '\n') FROM TABLE(FLATTEN(PARSE_JSON(:r):content)) c WHERE c.value:type = 'text');
END;
$$;

## Clean Up

When you're done, uncomment the two statements in this cell and run it to remove the database and warehouse. The domains are account-wide, so delete them separately from **Catalog > Business Ontology (Preview)**.

In [ ]:
%%sql -r clean_up
-- DROP DATABASE IF EXISTS BUSINESS_ONTOLOGY;
-- DROP WAREHOUSE IF EXISTS BUSINESS_ONTOLOGY_WH;
SELECT 'Uncomment the statements above to clean up' AS note;